# 예제 04. 데이터 증강
빅데이터프로그래밍 · 9주차

## 목표
- 회전 · 좌우 반전 · 확대축소 · 위치 이동을 눈으로 확인한다
- 학습 데이터에만 적용해야 하는 이유를 안다
- 과적합이 줄어드는 것을 비교한다

데이터를 늘릴 수 없을 때, 갖고 있는 데이터를 조금씩 바꿔 여러 장처럼 씁니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 네 가지 변형을 눈으로 보기


In [ ]:
raw = datasets.FashionMNIST("./data", train=True, download=True)
img, label = raw[0]        # PIL 이미지

augs = {
    "원본":       transforms.Compose([transforms.ToTensor()]),
    "회전 ±15°":  transforms.Compose([transforms.RandomRotation(15), transforms.ToTensor()]),
    "좌우 반전":  transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor()]),
    "확대·축소":  transforms.Compose([transforms.RandomAffine(0, scale=(0.7, 1.3)), transforms.ToTensor()]),
    "위치 이동":  transforms.Compose([transforms.RandomAffine(0, translate=(0.2, 0.2)), transforms.ToTensor()]),
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, (name, t) in zip(axes, augs.items()):
    ax.imshow(t(img).squeeze(), cmap="gray"); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# 같은 이미지에서 매번 다른 결과가 나옵니다 — 이것이 "여러 장처럼" 쓰는 방법
combo = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
])

fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for i, ax in enumerate(axes):
    ax.imshow(combo(img).squeeze(), cmap="gray"); ax.set_title(f"{i+1}", fontsize=10); ax.axis("off")
plt.suptitle("같은 이미지, 8번 변형", y=1.06)
plt.tight_layout(); plt.show()


## 2. 데이터에 맞는 변형을 골라야 합니다
좌우 반전은 옷에는 괜찮지만 **숫자에는 안 됩니다** — 2를 뒤집으면 2가 아닙니다.


In [ ]:
mnist = datasets.MNIST("./data", train=True, download=True)
digit, d_label = mnist[1]

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(transforms.ToTensor()(digit).squeeze(), cmap="gray")
axes[0].set_title(f"원본 ({d_label})"); axes[0].axis("off")
axes[1].imshow(transforms.RandomHorizontalFlip(p=1.0)(transforms.ToTensor()(digit)).squeeze(), cmap="gray")
axes[1].set_title("좌우 반전 — 부적절"); axes[1].axis("off")
axes[2].imshow(transforms.ToTensor()(transforms.RandomRotation(10)(digit)).squeeze(), cmap="gray")
axes[2].set_title("회전 10° — 괜찮음"); axes[2].axis("off")
axes[3].imshow(transforms.ToTensor()(transforms.RandomRotation(90)(digit)).squeeze(), cmap="gray")
axes[3].set_title("회전 90° — 과함"); axes[3].axis("off")
plt.tight_layout(); plt.show()


## 3. 학습 데이터에만 적용합니다
검증 데이터를 흔들면 매번 다른 점수가 나와 비교할 수 없습니다.


In [ ]:
train_tf = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
])
test_tf = transforms.ToTensor()          # 검증은 변형 없이

plain_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=test_tf)
aug_train   = datasets.FashionMNIST("./data", train=True,  download=True, transform=train_tf)
test_set    = datasets.FashionMNIST("./data", train=False, download=True, transform=test_tf)

plain_loader = DataLoader(Subset(plain_train, range(2000)), batch_size=64, shuffle=True)
aug_loader   = DataLoader(Subset(aug_train,   range(2000)), batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False)

print("학습 2,000장 · 검증 10,000장")


## 4. 학습 비교


In [ ]:
loss_fn = nn.CrossEntropyLoss()

class CNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.ReLU(),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.net(x)


def measure(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def train(loader, epochs=30, lr=1e-3):
    torch.manual_seed(42)
    model = CNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append((*measure(model, plain_loader), *measure(model, test_loader)))
    return model, hist


print("증강 없이");  plain, plain_hist = train(plain_loader)
print("증강 적용");  augm,  aug_hist   = train(aug_loader)

print("\n증강 없음  검증 정확도:", round(plain_hist[-1][3], 4), "· 차이", round(plain_hist[-1][1]-plain_hist[-1][3], 4))
print("증강 적용  검증 정확도:", round(aug_hist[-1][3], 4),   "· 차이", round(aug_hist[-1][1]-aug_hist[-1][3], 4))


In [ ]:
xs = range(1, len(plain_hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(xs, [h[2] for h in plain_hist], label="증강 없음")
ax[0].plot(xs, [h[2] for h in aug_hist],  label="증강 적용")
ax[0].set_title("검증 손실"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h[1]-h[3] for h in plain_hist], label="증강 없음")
ax[1].plot(xs, [h[1]-h[3] for h in aug_hist],  label="증강 적용")
ax[1].set_title("학습 − 검증 차이"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 5. 증강 강도를 바꿔 보기
너무 세게 흔들면 원래 모양을 잃어 오히려 나빠집니다.


In [ ]:
rows = []
for deg, name in [(0, "없음"), (10, "약함"), (25, "보통"), (60, "과함")]:
    tf = transforms.Compose([transforms.RandomRotation(deg), transforms.ToTensor()]) if deg else transforms.ToTensor()
    ds = datasets.FashionMNIST("./data", train=True, download=True, transform=tf)
    _, h = train(DataLoader(Subset(ds, range(2000)), batch_size=64, shuffle=True), epochs=20)
    rows.append({"회전 각도": deg, "설명": name, "검증 정확도": round(h[-1][3], 4)})
pd.DataFrame(rows)


## 직접 해보기
1. 좌우 반전만 적용하면 결과가 어떻게 되나요?
2. MNIST에 좌우 반전을 적용해 학습시키면 정확도가 어떻게 되나요? 왜 그런가요?


In [ ]:
# 여기에 작성하세요
